# Catalyst Prism 3B Upgrade
Qwen2.5-Coder-3B-Instruct → catalyst-prism-3b-q4_k_m.gguf

In [ ]:
%pip install -q -U unsloth "trl>=0.17.0" transformers datasets peft accelerate bitsandbytes sentencepiece safetensors

In [ ]:
import torch
from unsloth import FastLanguageModel
assert torch.cuda.is_available(), 'CUDA GPU required'
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
max_seq_length = 4096
model, tokenizer = FastLanguageModel.from_pretrained(model_name='Qwen/Qwen2.5-Coder-3B-Instruct', max_seq_length=max_seq_length, dtype=dtype, load_in_4bit=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
model = FastLanguageModel.get_peft_model(model, r=64, lora_alpha=128, lora_dropout=0.05, bias='none', target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], use_gradient_checkpointing='unsloth', random_state=3407)
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset
import hashlib
SYSTEM_PROMPT = '''You are Prism, Catalyst's reliable local coding model. Handle multi-file coding, debugging, scripts, and implementation. Ground claims in evidence. Never invent files, APIs, tests, or tool output. Never claim unrun tests passed.'''
def value(row, *names):
    for name in names:
        if row.get(name) is not None and str(row[name]).strip(): return str(row[name]).strip()
    return ''
def normalize(row):
    prompt=value(row,'instruction','prompt','question','problem'); context=value(row,'input','context','additional_context'); answer=value(row,'output','response','completion','answer')
    if context: prompt += '\n\nAdditional context:\n' + context
    return {'prompt':[{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':prompt}], 'completion':[{'role':'assistant','content':answer}]}
raw=load_dataset('sahil2801/CodeAlpaca-20k', split='train')
data=raw.map(normalize, remove_columns=raw.column_names)
data=data.filter(lambda r: len(r['prompt'][1]['content'])>=16 and 30<=len(r['completion'][0]['content'])<=16000)
data=data.map(lambda r: {'hash': hashlib.sha256((r['prompt'][1]['content']+'\n'+r['completion'][0]['content']).encode()).hexdigest()})
seen=set()
data=data.filter(lambda r: False if r['hash'] in seen else (seen.add(r['hash']) is None or True)).remove_columns('hash')
split=data.train_test_split(test_size=0.05, seed=3407)
train_dataset, eval_dataset = split['train'], split['test']
print(len(train_dataset), len(eval_dataset))

In [ ]:
from trl import SFTTrainer, SFTConfig
args=SFTConfig(output_dir='prism_3b_outputs', num_train_epochs=2, per_device_train_batch_size=2 if gpu_memory_gb>=30 else 1, per_device_eval_batch_size=1, gradient_accumulation_steps=8, learning_rate=8e-5, lr_scheduler_type='cosine', warmup_ratio=0.05, weight_decay=0.01, optim='adamw_8bit', max_grad_norm=1.0, bf16=dtype==torch.bfloat16, fp16=dtype==torch.float16, max_length=max_seq_length, packing=True, completion_only_loss=True, eval_strategy='steps', eval_steps=100, save_strategy='steps', save_steps=100, save_total_limit=3, load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False, logging_steps=10, report_to='none', seed=3407, data_seed=3407)
trainer=SFTTrainer(model=model, processing_class=tokenizer, train_dataset=train_dataset, eval_dataset=eval_dataset, args=args)
print(trainer.train().metrics)
print(trainer.evaluate())
model.save_pretrained('catalyst_prism_3b_lora')
tokenizer.save_pretrained('catalyst_prism_3b_lora')
model.save_pretrained_gguf('catalyst_prism_3b_q4', tokenizer, quantization_method='q4_k_m')

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive', force_remount=False)
files=list(Path('/content').glob('**/catalyst_prism_3b_q4*.gguf'))
assert files, 'Prism GGUF not found'
source=max(files, key=lambda path: path.stat().st_size)
destination_dir=Path('/content/drive/MyDrive/Catalyst/models')
destination_dir.mkdir(parents=True, exist_ok=True)
destination=destination_dir/'catalyst-prism-3b-q4_k_m.gguf'
shutil.copy2(source, destination)
assert source.stat().st_size == destination.stat().st_size
print(f'Copied to: {destination}')
print(f'Size: {destination.stat().st_size / 2**30:.2f} GB')